# 01 — Quickstart: Connect & First Query

This notebook assumes you already have the Fractal RAG stack running
(either the [local docker-compose stack](../local-setup.md) on `localhost:8080`
or the [K3s deployment](../hosted-vm.md) on `localhost:30080`) and that
`fractal-rag ingest` has loaded at least the `docs` and `notebooks` sources.

By the end of this notebook you'll have:

1. Connected to Weaviate with the same Settings the CLI uses.
2. Confirmed the schema and concept ontology are loaded.
3. Run a hybrid search and inspected one full artifact.

In [ ]:
# If you used docker-compose locally, uncomment these:
# import os
# os.environ['FRACTAL_RAG_WEAVIATE_HTTP_PORT'] = '8080'
# os.environ['FRACTAL_RAG_WEAVIATE_GRPC_PORT'] = '50051'

from fractal_rag.client import client_session
from fractal_rag.config import get_settings

settings = get_settings(refresh=True)
print(f'Targeting Weaviate at {settings.weaviate_http_host}:{settings.weaviate_http_port}')

## Sanity check

Make sure both collections exist and have objects.

In [ ]:
from fractal_rag.client import client_session

with client_session() as client:
    for name in (settings.artifact_collection, settings.concept_collection):
        if not client.collections.exists(name):
            print(f'{name}: MISSING')
            continue
        coll = client.collections.get(name)
        total = coll.aggregate.over_all(total_count=True).total_count
        print(f'{name}: {total} objects')

## Browse the concept ontology

The `Concept` collection holds ~31 hand-curated entries. Listing them is the fastest way to orient yourself.

In [ ]:
with client_session() as client:
    concept = client.collections.get(settings.concept_collection)
    response = concept.query.fetch_objects(limit=50)
    for o in response.objects:
        p = o.properties
        print(f"- {p['name']}  ({len(p.get('aliases') or [])} aliases) - domains: {p.get('domain')}")

## Your first hybrid search

`coll.query.hybrid(...)` mixes BM25 keyword scoring with vector similarity. `target_vector` picks **which** of the named vectors to compare against. Try `content_vec` (the body), `title_vec` (just the title), or `summary_vec` (the short summary).

In [ ]:
from weaviate.classes.query import MetadataQuery

QUERY = 'Hausdorff dimension of the Cantor set'

with client_session() as client:
    art = client.collections.get(settings.artifact_collection)
    response = art.query.hybrid(
        query=QUERY,
        limit=5,
        target_vector='content_vec',
        return_metadata=MetadataQuery(distance=True),
    )

for o in response.objects:
    p = o.properties
    d = getattr(o.metadata, 'distance', None)
    print(f"[{d:.3f}] {p.get('source_type')}  {p.get('title')[:80]}")

## Pull the full content of a hit

The hybrid response only returns the headline fields. If you want to read the actual chunk content, fetch by UUID.

In [ ]:
best = response.objects[0]
with client_session() as client:
    art = client.collections.get(settings.artifact_collection)
    full = art.query.fetch_object_by_id(str(best.uuid))

print(f"TITLE   : {full.properties.get('title')}")
print(f"URI     : {full.properties.get('source_uri')}")
print(f"SUMMARY : {full.properties.get('summary')}")
print('CONTENT (first 800 chars):')
print(full.properties.get('content', '')[:800])

## Next steps

- Try [02 — Hybrid search patterns](02_hybrid_search.ipynb) for `target_vector` tradeoffs, `alpha` tuning, and filters.
- Or skip to [03 — Concept ontology](03_concept_ontology.ipynb) to walk the Concept side-collection and its cross-refs.